In [ ]:
# install or update libs (if necessary!)
#!pip install datasets
#!pip install --upgrade torch
#!pip install --upgrade accelerate
##!pip install --upgrade transformers[torch]

In [2]:
# import libraries

# main libs
import numpy as np
import pandas as pd
from tqdm import tqdm
import random
import os
import warnings

# data
from datasets import load_dataset

# sklearn for evaluations
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score

# transformers
import torch
import torch.nn as nn
import transformers
import accelerate
#from transformers.file_utils import is_tf_available, is_torch_available # utils
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer # models & tokenisers
from transformers import EarlyStoppingCallback # to set early stopping
from transformers import TrainingArguments, Trainer # load training libs

In [3]:
# disable WANDB_MODE -- important!
import os
os.environ["WANDB_MODE"] = "disabled"

In [4]:
# Suppress the warning
warnings.filterwarnings("ignore",
                        message="Repo card metadata block was not found.",
                        category=UserWarning)

In [5]:
# Check for GPU availability
torch.cuda.is_available()

False

In [6]:
# check correct import
print("Pytorch version:", torch.__version__) # pytorch 2.1.0+

Pytorch version: 2.11.0+cpu


In [7]:
# Set device
device = torch.device('cpu') # cpu for windows, 'mps' for mac
print(device)

cpu


In [9]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # To force: force_remount=True

Mounted at /content/drive


In [10]:
# Set random seed
seed = torch.randint(0, 1000, (1,)).item()
seed # record for replication

850

## Load data

In [19]:
# import df
dataset = load_dataset("Jacobvs/PoliticalTweets")

# Load the main split into a pandas DataFrame
split_name = list(dataset.keys())[0]
df_raw = dataset[split_name].to_pandas()

In [14]:
df_raw

,index,date,id,username,text,party,labels
0,0,2021-10-13 19:47:44,1448374915636383745,SenatorHassan,Happy th birthday to the @USNavy! The strength...,Democrat,1
1,1,2021-06-30 14:53:13,1410250073003462656,SenatorMenendez,The greatest generation's investment in infras...,Democrat,1
2,2,2021-08-08 01:11:29,1424176405881966599,SenBillCassidy,"Thanks to @SenTedCruz and @SenatorWarnock, th...",Republican,0
3,3,2021-04-14 14:02:49,1382333523567185921,SenBlumenthal,/ To get lasting change we cant just lock up t...,Democrat,1
4,4,2021-12-11 16:06:38,1469700160934621188,SenatorBraun,Today were celebrating years of the Hoosier st...,Republican,0
...,...,...,...,...,...,...,...
190486,191475,2022-05-13 15:00:51,1525128949583462402,RepWalberg,Religious liberty is foundational to America. ...,Republican,0
190487,191476,2022-05-12 20:36:47,1524851099211120640,RepWalberg,"During #NationalPoliceWeek, I want to give a h...",Republican,0
190488,191477,2022-05-12 17:47:11,1524808421488336896,RepWalberg,The same week that gas prices hit a record hig...,Republican,0
190489,191478,2022-05-12 15:26:15,1524772953501712384,RepWalberg,Amid another mounting crisis—a nationwide shor...,Republican,0


In [27]:
# remove cols
df = df_raw.drop(columns= ['index'])
df

,date,id,username,text,party,labels
0,2021-10-13 19:47:44,1448374915636383745,SenatorHassan,Happy th birthday to the @USNavy! The strength...,Democrat,1
1,2021-06-30 14:53:13,1410250073003462656,SenatorMenendez,The greatest generation's investment in infras...,Democrat,1
2,2021-08-08 01:11:29,1424176405881966599,SenBillCassidy,"Thanks to @SenTedCruz and @SenatorWarnock, th...",Republican,0
3,2021-04-14 14:02:49,1382333523567185921,SenBlumenthal,/ To get lasting change we cant just lock up t...,Democrat,1
4,2021-12-11 16:06:38,1469700160934621188,SenatorBraun,Today were celebrating years of the Hoosier st...,Republican,0
...,...,...,...,...,...,...
190486,2022-05-13 15:00:51,1525128949583462402,RepWalberg,Religious liberty is foundational to America. ...,Republican,0
190487,2022-05-12 20:36:47,1524851099211120640,RepWalberg,"During #NationalPoliceWeek, I want to give a h...",Republican,0
190488,2022-05-12 17:47:11,1524808421488336896,RepWalberg,The same week that gas prices hit a record hig...,Republican,0
190489,2022-05-12 15:26:15,1524772953501712384,RepWalberg,Amid another mounting crisis—a nationwide shor...,Republican,0


In [40]:
# only sample 1-4 scores
df_filtered = df[df['labels'].isin([0,1])]

In [41]:
# make labels binary
#score_mapping = {
#    1: 0,
#    2: 0,
#    3: 0,
#    4: 1
#    }
#df_filtered['label'] = df_filtered['score'].map(score_mapping)

In [42]:
# let's check again!
df_filtered.labels.value_counts()

,count
labels,
1,97901
0,92590


In [43]:
# Remove possible duplicates
df_filtered = df_filtered.drop_duplicates(subset=['text'], keep='first')
df_filtered.rename(columns={'labels': 'label'}, inplace=True)
len(df)

190491

In [44]:
# Set label as int, text as str
df_filtered['label'] = df_filtered['label'].astype('int')
df_filtered['text'] = df_filtered['text'].astype(str)

# Set label to category
df_filtered['label'] = pd.Categorical(df_filtered.label)

In [45]:
# Subset
df_filtered=df_filtered[['text', 'label']]

In [64]:
# Set DEMO df
df_demo = df_filtered.sample(frac=0.0003, random_state=42) ## small fraction
len(df_demo)

57

## Train/Val/Test sets

In [65]:
# Set lists
texts = list(df_demo['text'])
labels = list(df_demo['label'])

In [66]:
# Split the data in training and remaining dataset
x_train, x_rem, y_train, y_rem = train_test_split(texts, labels, test_size=0.2,
                                                  random_state= seed,
                                                  stratify = labels)

# Split remaining data in testing and validation sets
x_val, x_test, y_val, y_test = train_test_split(x_rem,y_rem, test_size=0.5,
                                                random_state= seed)

# Check lens!
print(len(x_train), len(x_val), len(x_test))

45 6 6


## Tokenisation

In [67]:
# Load tokenizer to process the text field:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased",
                                          use_fast=True)

In [68]:
# Apply tokenisation
train_encodings = tokenizer(x_train, truncation=True, padding=True)
val_encodings = tokenizer(x_val, truncation=True, padding=True)
test_encodings = tokenizer(x_test, truncation=True, padding=True)

In [69]:
# Set full datasets tokenised text & labels
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, y_train)
val_dataset = Dataset(val_encodings, y_val)
test_dataset = Dataset(test_encodings, y_test)

## Fine-tuning

In [70]:
# Count the number of unique labels
nr_labels = df_demo['label'].nunique()
print(nr_labels)

2


In [71]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", # see full list of models on huggingface.co
                                                           num_labels=nr_labels)
model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [72]:
# Compute loss function
def compute_loss(model, inputs, labels):
    # Forward pass through the model
    outputs = model(**inputs)

    # Get the logits from the model outputs and convert them to float data type
    logits = outputs.logits.float()

    # Define the loss function (see comment: with class weights)
    loss_fn = nn.CrossEntropyLoss() #weight=class_weights

    # Compute the loss using the defined loss function
    loss = loss_fn(logits, labels)

    return loss  # Return the computed loss

In [73]:
# Defining the TrainingArguments() arguments
args = TrainingArguments(
    output_dir=f"/content/drive/MyDrive/results_demo_{seed}", ## change based on your drive
    seed=seed,
    do_train=True,
    do_eval=True,
    eval_strategy='steps',  # Evaluate every N steps
    eval_steps=50,
    save_strategy='steps',  # RECOMMENDED: Save checkpoints at regular intervals
    save_steps=50,  # Save every 50 steps (aligned with eval_steps)
    save_total_limit=5,
    load_best_model_at_end=True,  # UNCOMMENT THIS for early stopping
    metric_for_best_model='eval_loss',  # Use 'eval_loss' not 'loss'
    greater_is_better=False,  # Lower loss is better
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    push_to_hub=False,
)

In [74]:
# Define metrics computation function
def compute_metrics(p):
    # Unpack the predictions and labels
    pred, labels = p

    # Convert predictions to class labels
    pred = np.argmax(pred, axis=1)

    # Compute accuracy
    accuracy = accuracy_score(y_true=labels, y_pred=pred)

    # Compute F1 score
    f1 = f1_score(y_true=labels, y_pred=pred, average='weighted')

    # Return a dictionary containing computed metrics
    return {"accuracy": accuracy, "f1": f1}

In [75]:
# Set Trainer
trainer = Trainer(
    model=model,  # Pass the model to the Trainer
    args=args,  # Pass the training arguments to the Trainer
    train_dataset=train_dataset,  # Pass the training dataset to the Trainer
    eval_dataset=val_dataset,  # Pass the validation dataset to the Trainer
    #tokenizer=tokenizer,  # Pass the tokenizer to the Trainer
    compute_metrics=compute_metrics,  # Specify the function to compute metrics
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # Specify callbacks for training (eg, early stopping)
)

In [ ]:
# Run trainer (takes a while)
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


## Model Evaluation

In [ ]:
# Evaluation
trainer.evaluate()

In [ ]:
# Quick evaluation on test set
trainer.evaluate(test_dataset)

In [ ]:
# Predict test set (we use the results below)
predict = trainer.predict(test_dataset)

In [ ]:
# Define preds/labels for validation
preds = predict.predictions
labels = predict.label_ids

In [ ]:
# Create f-score metric
def f1_score_func(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return f1_score(labels_flat, preds_flat, average='weighted')

In [ ]:
# Compute the predicted class for each sample by finding the index of the maximum value along axis 1 of the preds array
preds_flat = np.argmax(preds, axis=1).flatten()

# Flatten the labels array to ensure it has the same shape as preds_flat
labels_flat = labels.flatten()

In [ ]:
# Plot confusion matrix
print(confusion_matrix(labels_flat, preds_flat))
print(classification_report(labels_flat, preds_flat))

In [ ]:
# AUC test (Goal: >0.7, yet it depends on task)
fpr, tpr, thresholds = metrics.roc_curve(labels_flat, preds_flat, pos_label=1) # set pos_labels to nr of labels -1
metrics.auc(fpr, tpr)

## Save

In [ ]:
# Save
trainer.save_model(output_dir=f"/content/drive/MyDrive/results_demo_{seed}") ## change based on your drive
trainer.save_state()

In [ ]:
# Save the complete model (this should create pytorch_model.bin or model.safetensors)
save_path = f"/content/drive/MyDrive/results_demo_{seed}" ## change based on your drive
trainer.save_model(save_path)

# Also save the tokenizer
tokenizer.save_pretrained(save_path)

# Verify the files were saved
import os
print("Files saved:")
print(os.listdir(save_path))